In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import os
import yfinance as yf

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping



warnings.simplefilter(action="ignore", category=FutureWarning)

In [3]:
# =====================================================================
# 1. DESCARGA Y PREPARACIÓN DE DATOS (Código del Profesor)
# =====================================================================
print("Descargando datos de Yahoo Finance...")
start_date = '1945-01-01'
tickers_validos = ['AEP', 'BA', 'CAT', 'CNP', 'CVX', 'DIS', 'DTE', 'ED', 'GD', 'GE', 
                   'HON', 'HPQ', 'IBM', 'IP', 'JNJ', 'KO', 'KR', 'MMM', 'MO', 'MRK', 
                   'MSI', 'PG', 'XOM']

precios_close = yf.download(tickers_validos, start=start_date, auto_adjust=True, progress=False)['Close']
precios_close.dropna(axis=1, inplace=True)

# Cálculo de retornos logarítmicos
returns = np.log(precios_close).diff().dropna()
print(f"Forma de los datos de retornos: {returns.shape}")

# Función del profesor para crear ventanas
def create_time_series_data(data, input_window_size, output_window_size):
    X, y = [], []
    data_array = data.values if isinstance(data, pd.DataFrame) else data
    num_features = data_array.shape[1] 

    for i in range(len(data_array) - input_window_size - output_window_size + 1):
        input_sequence = data_array[i : i + input_window_size]
        X.append(input_sequence)
        
        if output_window_size > 0:
            output_sequence = data_array[i + input_window_size : i + input_window_size + output_window_size]
            average_output = np.mean(output_sequence, axis=0) 
            y.append(average_output)
        else:
            y.append(data_array[i + input_window_size - 1])
            
    return np.array(X), np.array(y)

Descargando datos de Yahoo Finance...


c:\Users\Joseph\AppData\Local\Programs\Python\Python313\Lib\site-packages\yfinance\scrapers\history.py:201: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
c:\Users\Joseph\AppData\Local\Programs\Python\Python313\Lib\site-packages\yfinance\scrapers\history.py:144: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end_dt = pd.Timestamp.utcnow().tz_convert(tz)
c:\Users\Joseph\AppData\Local\Programs\Python\Python313\Lib\site-packages\yfinance\scrapers\history.py:144: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end_dt = pd.Timestamp.utcnow().tz_convert(tz)
c:\Users\Joseph\AppData\Local\Programs\Python\Python313\Lib\site-packages\yfinance\scrapers\history.py:201: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a f

Forma de los datos de retornos: (16189, 23)


In [11]:
# =====================================================================
# 2. DEFINICIÓN DE ARQUITECTURA Y BASELINES
# =====================================================================

def construir_modelo_rnn(config, input_shape, n_assets=23):
    """Construye un modelo dinámico basado en la configuración dada."""
    model = Sequential()
    model.add(Input(shape=input_shape))
    
    # Capa dinámica (LSTM o GRU)
    CapaRecurrente = config['tipo_capa']
    model.add(CapaRecurrente(config['neuronas'], return_sequences=False))
    
    model.add(Dropout(config['dropout']))
    model.add(Dense(n_assets)) # Salida: 23 valores (promedio de los 23 activos)
    
    optimizador = Adam(learning_rate=config['lr'])
    model.compile(optimizer=optimizador, loss='mae')
    return model

def calcular_baselines(X_test, y_test, y_train_mean):
    """Calcula el MAE para modelos simples, incluyendo Buy and Hold."""
    
    # 1. Baseline Naive: El futuro será igual al último día de la ventana de entrada
    y_pred_naive = X_test[:, -1, :]
    mae_naive = np.mean(np.abs(y_pred_naive - y_test))
    
    # 2. Baseline SMA: El futuro será igual a la media de la ventana de entrada actual
    y_pred_sma = np.mean(X_test, axis=1)
    mae_sma = np.mean(np.abs(y_pred_sma - y_test))
    
    # 3. Baseline Buy and Hold: Predecir siempre la media histórica del entrenamiento
    # Creamos un array del mismo tamaño que y_test relleno con la media de y_train
    y_pred_bh = np.full_like(y_test, y_train_mean)
    mae_bh = np.mean(np.abs(y_pred_bh - y_test))
    
    return mae_naive, mae_sma, mae_bh

# Crear carpeta para guardar gráficas si no existe
os.makedirs('graficas_convergencia', exist_ok=True)

In [16]:
# =====================================================================
# 3. CONFIGURACIÓN DEL EXPERIMENTO
# =====================================================================

input_windows = [5, 10, 30, 90]
output_windows = [1, 5, 30, 90]

# Espacio de búsqueda de hiperparámetros (puedes añadir o quitar)
lista_hiperparametros = [
    # 1. El "Estándar" (para ver si una GRU con más neuronas supera a la LSTM)
    {'tipo_capa': GRU,  'neuronas': 64, 'dropout': 0.1, 'lr': 0.001},
    
    # 2. El "Arquitecto" (mayor capacidad para ventanas difíciles de 1 y 5 días)
    {'tipo_capa': LSTM, 'neuronas': 128, 'dropout': 0.2, 'lr': 0.0005},
    
    # 3. El "Minucioso" (pasos muy pequeños para evitar la curva en L)
    {'tipo_capa': LSTM, 'neuronas': 64, 'dropout': 0.1, 'lr': 0.0001}
]
# Matrices para reportar resultados finales
matriz_mae_rnn = np.zeros((4, 4))
matriz_mae_naive = np.zeros((4, 4))
matriz_mae_sma = np.zeros((4, 4))
matriz_mae_bh = np.zeros((4, 4))

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)


In [17]:
# =====================================================================
# 4. BUCLE PRINCIPAL (AUTOMATIZACIÓN DE LOS 16 MODELOS x CONFIGURACIONES)
# =====================================================================

print("\nIniciando entrenamiento de modelos...")

for i, in_w in enumerate(input_windows):
    for j, out_w in enumerate(output_windows):
        print(f"\n=======================================================")
        print(f" Ventana Entrada: {in_w} días | Ventana Salida: {out_w} días")
        print(f"=======================================================")
        
        # 1. Crear datos
        X, y = create_time_series_data(returns, in_w, out_w)
        
        # 2. Separación CRONOLÓGICA: 80% Train, 10% Validacion, 10% Test
        split_1 = int(len(X) * 0.8)
        split_2 = int(len(X) * 0.9)
        
        X_train, y_train = X[:split_1], y[:split_1]
        X_val, y_val = X[split_1:split_2], y[split_1:split_2]
        X_test, y_test = X[split_2:], y[split_2:]

        # Calculamos la media global de entrenamiento para esta ventana (Buy and Hold)
        y_train_mean = np.mean(y_train, axis=0)
        
        # 3. Baselines
        mae_naive, mae_sma, mae_bh = calcular_baselines(X_test, y_test, y_train_mean)
        matriz_mae_naive[i, j] = mae_naive
        matriz_mae_sma[i, j] = mae_sma
        matriz_mae_bh[i, j] = mae_bh
        print(f"Baseline Naive (MAE en Test): {mae_naive:.6f}")
        print(f"Baseline SMA   (MAE en Test): {mae_sma:.6f}")
        print(f"Baseline Buy & Hold (MAE): {mae_bh:.6f}")

        
        # 4. Búsqueda del mejor modelo recurrente
        mejor_val_loss = float('inf')
        mejor_modelo = None
        mejor_historial = None
        mejor_config = None
        
        for config in lista_hiperparametros:
            capa_nombre = config['tipo_capa'].__name__
            print(f" -> Entrenando: {capa_nombre}, Neuronas: {config['neuronas']}, LR: {config['lr']}...")
            
            modelo = construir_modelo_rnn(config, input_shape=(in_w, 23))
            
            # Usamos verbose=0 para no llenar la pantalla de números, epochs=50 es suficiente con EarlyStop
            historial = modelo.fit(X_train, y_train, 
                                   validation_data=(X_val, y_val),
                                   epochs=50, 
                                   batch_size=64, 
                                   callbacks=[early_stop], 
                                   verbose=0)
            
            val_loss_actual = min(historial.history['val_loss'])
            
            if val_loss_actual < mejor_val_loss:
                mejor_val_loss = val_loss_actual
                mejor_modelo = modelo
                mejor_historial = historial
                mejor_config = config
        
        print(f"\n[GANADOR] {mejor_config['tipo_capa'].__name__} ({mejor_config['neuronas']} neuronas)")
        
        # 5. Evaluación final del GANADOR en TEST
        mae_test_ganador = mejor_modelo.evaluate(X_test, y_test, verbose=0)
        matriz_mae_rnn[i, j] = mae_test_ganador
        print(f"MAE del Modelo Ganador en TEST: {mae_test_ganador:.6f}")
        
        # 6. Guardar Gráfica de Convergencia del Ganador
        plt.figure(figsize=(10, 5))
        plt.plot(mejor_historial.history['loss'], label='Error Entrenamiento (MAE)')
        plt.plot(mejor_historial.history['val_loss'], label='Error Validación (MAE)')
        
        # Título con todos los hiperparámetros
        nombre_capa = mejor_config['tipo_capa'].__name__
        n_neuronas = mejor_config['neuronas']
        l_rate = mejor_config['lr']
        d_out = mejor_config['dropout']
        
        plt.title(f"Convergencia {nombre_capa} | Neuronas: {n_neuronas} | LR: {l_rate} | Drop: {d_out}\n(Ventana In:{in_w} - Out:{out_w})")
        
        plt.xlabel('Épocas')
        plt.ylabel('MAE')
        plt.legend()
        plt.grid(True)
        
        # Guardar la imagen
        nombre_archivo = f"graficas_convergencia/conver_in{in_w}_out{out_w}.png"
        plt.savefig(nombre_archivo)
        plt.close()


Iniciando entrenamiento de modelos...

 Ventana Entrada: 5 días | Ventana Salida: 1 días
Baseline Naive (MAE en Test): 0.017773
Baseline SMA   (MAE en Test): 0.013604
Baseline Buy & Hold (MAE): 0.012240
 -> Entrenando: GRU, Neuronas: 64, LR: 0.001...
 -> Entrenando: LSTM, Neuronas: 128, LR: 0.0005...
 -> Entrenando: LSTM, Neuronas: 64, LR: 0.0001...

[GANADOR] LSTM (64 neuronas)
MAE del Modelo Ganador en TEST: 0.012266

 Ventana Entrada: 5 días | Ventana Salida: 5 días
Baseline Naive (MAE en Test): 0.013655
Baseline SMA   (MAE en Test): 0.008028
Baseline Buy & Hold (MAE): 0.005581
 -> Entrenando: GRU, Neuronas: 64, LR: 0.001...
 -> Entrenando: LSTM, Neuronas: 128, LR: 0.0005...
 -> Entrenando: LSTM, Neuronas: 64, LR: 0.0001...

[GANADOR] LSTM (128 neuronas)
MAE del Modelo Ganador en TEST: 0.005616

 Ventana Entrada: 5 días | Ventana Salida: 30 días
Baseline Naive (MAE en Test): 0.012519
Baseline SMA   (MAE en Test): 0.006164
Baseline Buy & Hold (MAE): 0.002320
 -> Entrenando: GRU, Neu

In [18]:
# =====================================================================
# 5. RESULTADOS FINALES (Tablas para tu GitHub y Presentación)
# =====================================================================

print("\n\n" + "="*50)
print("MATRIZ DE RESULTADOS FINALES EN TEST (REDES RECURRENTES)")
print("="*50)
df_rnn = pd.DataFrame(matriz_mae_rnn, 
                      index=[f'In_{w}' for w in input_windows], 
                      columns=[f'Out_{w}' for w in output_windows])
print(df_rnn)

print("\n" + "="*50)
print("MATRIZ DE RESULTADOS BASELINE NAIVE (PARA COMPARAR)")
print("="*50)
df_naive = pd.DataFrame(matriz_mae_naive, 
                        index=[f'In_{w}' for w in input_windows], 
                        columns=[f'Out_{w}' for w in output_windows])
print(df_naive)

print("MATRIZ DE RESULTADOS BASELINE SMA (PARA COMPARAR)")
print("="*50)
df_sma = pd.DataFrame(matriz_mae_sma, 
                        index=[f'In_{w}' for w in input_windows], 
                        columns=[f'Out_{w}' for w in output_windows])
print(df_sma)

print("MATRIZ DE RESULTADOS BASELINE BUY AND HOLD (PARA COMPARAR)")
print("="*50)
df_bh = pd.DataFrame(matriz_mae_bh, 
                        index=[f'In_{w}' for w in input_windows], 
                        columns=[f'Out_{w}' for w in output_windows])
print(df_bh)



MATRIZ DE RESULTADOS FINALES EN TEST (REDES RECURRENTES)
          Out_1     Out_5    Out_30    Out_90
In_5   0.012266  0.005616  0.002338  0.001282
In_10  0.012339  0.005661  0.002396  0.001337
In_30  0.012478  0.005651  0.002378  0.001774
In_90  0.012341  0.005658  0.002399  0.001380

MATRIZ DE RESULTADOS BASELINE NAIVE (PARA COMPARAR)
          Out_1     Out_5    Out_30    Out_90
In_5   0.017773  0.013655  0.012519  0.012253
In_10  0.017779  0.013655  0.012518  0.012255
In_30  0.017783  0.013659  0.012524  0.012259
In_90  0.017816  0.013676  0.012528  0.012250
MATRIZ DE RESULTADOS BASELINE SMA (PARA COMPARAR)
          Out_1     Out_5    Out_30    Out_90
In_5   0.013604  0.008028  0.006164  0.005780
In_10  0.012947  0.006909  0.004737  0.004229
In_30  0.012515  0.006164  0.003490  0.002719
In_90  0.012346  0.005799  0.002755  0.001824
MATRIZ DE RESULTADOS BASELINE BUY AND HOLD (PARA COMPARAR)
          Out_1     Out_5   Out_30    Out_90
In_5   0.012240  0.005581  0.00232  0.001265